# Lab 01 - Explore Microsoft Web IQ APIs

**Goal:** retrieve fresh public evidence, inspect sources, and choose the right API for a grounding task. Allow 40 minutes.

Web IQ is a web intelligence service, not an LLM. Its Azure resource provides billing; API requests go to `https://api.microsoft.ai/v3`. We will start with search passages, then explore localization and account-dependent services.

**Before running:** deploy the Bicep infrastructure, link `WEBIQ_RESOURCE_ID` in [Profile Management](https://webiq.microsoft.ai/profiles), create an API key, and select the workshop Python kernel. See the root README for installation. Only use public, non-sensitive queries. Calls incur usage charges; avoid repeatedly running all cells.

**Outcomes:** explain passages versus full documents, distinguish crawl dates from publication dates, retain source attribution, handle crawl/retry responses, inspect structured place information and Auto evidence, and prepare an evidence-backed briefing.

## 1. Configuration and authentication

Settings come from the deployment's `.env.azure`, then your optional `.env`. Existing environment variables take precedence. A hidden prompt avoids placing a key in notebook source or outputs.

This lab uses a **Web IQ profile API key** in `x-apikey`, not an Azure resource key. Production Entra authentication requires an app-only token for `https://api.microsoft.ai/.default` and a client ID bound to the Web IQ profile; a developer's `az login` token is not a substitute.

In [ ]:
import getpass
import json
import os
import time
from pathlib import Path

import requests
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open this notebook from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')

api_key = os.getenv('WEBIQ_API_KEY') or getpass.getpass('Web IQ API key: ')
if not api_key.strip():
    raise ValueError('A Web IQ portal API key is required.')
print('Configuration loaded. The API key will not be displayed.')

## 2. Read the API calls directly

Each exercise shows its complete `requests.post` call: the endpoint, authentication header, JSON body and timeout. `raise_for_status()` surfaces HTTP errors, and printing the response JSON lets you inspect the actual fields, including `traceId` for support.

There is no request wrapper or automatic retry for search calls. If you receive HTTP 429 or 503, inspect `response.headers.get('Retry-After')`, wait as directed, and rerun that cell. Browse shows its bounded 202 polling explicitly in its own exercise. The small configuration check below only controls optional exercises; it does not grant permissions.

In [ ]:
def enabled(service):
    value = os.getenv(f'WEBIQ_ENABLE_{service.upper()}', 'false').lower()
    if value not in ('true', 'false'):
        raise ValueError(f'WEBIQ_ENABLE_{service.upper()} must be true or false.')
    if value == 'false':
        print(f'{service}: not run. Confirm profile access, then set WEBIQ_ENABLE_{service.upper()}=true and restart the kernel.')
    return value == 'true'

## 3. Search the web for grounding passages

The documented starting point is `maxResults=10`, `contentFormat='passage'`, `maxLength=3000`. A passage is query-relevant evidence, not a generated answer or a full webpage. Web Search does **not** return a `snippet` field.

Inspect titles, URLs, content and optional timestamps. A successful response may omit `webResults` when there are no matches. Never invent missing dates or sources.

In [ ]:
question = 'What are the latest Microsoft announcements about AI agents?'
response = requests.post(
    'https://api.microsoft.ai/v3/search/web',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': question, 'maxResults': 10,
        'contentFormat': 'passage', 'maxLength': 3000,
        'language': 'en', 'region': 'US', 'safeSearch': 'strict',
    },
    timeout=(10, 90),
)
response.raise_for_status()
web = response.json()
print(json.dumps(web, indent=2, ensure_ascii=False))

**Checkpoint:** which two sources would you trust for the question, and why? Does `crawledAt` mean publication date? Could the first-ranked source still be outdated?

## 4. Localization and deliberate source scoping

Compare the English/US search with Spanish/Spain. The language parameter controls interface language; it does not guarantee that every returned document is in that language. A `site:` operator narrows recall and should be used only when the task needs it. `safeSearch='strict'` is not an absolute content-safety guarantee, especially with `site:`.

In [ ]:
response = requests.post(
    'https://api.microsoft.ai/v3/search/web',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': 'novedades de agentes de inteligencia artificial de Microsoft site:learn.microsoft.com',
        'language': 'es', 'region': 'ES', 'safeSearch': 'strict',
        'maxResults': 10, 'contentFormat': 'passage', 'maxLength': 3000,
    },
    timeout=(10, 90),
)
response.raise_for_status()
localized = response.json()
print(json.dumps(localized, indent=2, ensure_ascii=False))

## 5. Browse: from a discovered source to page content (optional entitlement)

Search starts with a question; Browse starts with a URL. `liveCrawl='fallback'` requests a crawl when indexed content is unavailable. A 202 means processing is still underway. A forced crawl can return stale content with `liveCrawlSucceeded=false`; absence of that flag is **not** an explicit freshness guarantee.

Full-document access may require separate enablement. Set `WEBIQ_ENABLE_BROWSE=true` only when entitled. This cell selects a public search result and requests Markdown, showing plain text rather than rendering untrusted HTML.

The loop makes up to four requests. For a pending crawl it waits for the returned `retryAfter` duration, such as `60s`. Missing delays, waits longer than two minutes, and unfinished crawls raise explicit errors instead of being treated as page content.

In [ ]:
if enabled('browse'):
    sources = web.get('webResults', [])
    if not sources or not sources[0].get('url'):
        raise RuntimeError('No source URL to browse. Change the search query and rerun search.')
    for attempt in range(4):
        response = requests.post(
            'https://api.microsoft.ai/v3/browse',
            headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
            json={
                'url': sources[0]['url'], 'contentFormat': 'markdown',
                'maxLength': 3000, 'liveCrawl': 'fallback',
            },
            timeout=(10, 90),
        )
        response.raise_for_status()
        page = response.json()
        print(json.dumps(page, indent=2, ensure_ascii=False))
        if response.status_code == 200:
            break
        if response.status_code != 202:
            raise RuntimeError(f'Unexpected Browse status: {response.status_code}.')
        if attempt == 3:
            raise TimeoutError('Browse did not complete after four requests. Retry later.')
        if not page.get('retryAfter'):
            raise RuntimeError('Pending crawl omitted retryAfter.')
        seconds = float(page['retryAfter'].removesuffix('s'))
        if not 0 <= seconds <= 120:
            raise TimeoutError(f'Server requested a {seconds}s wait; retry later manually.')
        print(f'Crawl pending. Waiting {seconds:g}s...')
        time.sleep(seconds)

## 6. News: publisher attribution and current events (optional entitlement)

Set `WEBIQ_ENABLE_NEWS=true` when entitled. News results can include `source`, `lastUpdatedAt`, `crawledAt` and a thumbnail, but these fields are optional. Compare the publishers and date coverage against Web Search. News Search has a maximum of 20 results; do not assume it supports the video endpoint's `freshness` filter.

In [ ]:
if enabled('news'):
    response = requests.post(
        'https://api.microsoft.ai/v3/search/news',
        headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
        json={
            'query': question, 'maxResults': 5, 'language': 'en', 'region': 'US',
            'contentFormat': 'passage', 'maxLength': 3000,
        },
        timeout=(10, 90),
    )
    response.raise_for_status()
    news = response.json()
    print(json.dumps(news, indent=2, ensure_ascii=False))

## 7. Image and video discovery (optional entitlements)

Set `WEBIQ_ENABLE_IMAGES=true` and/or `WEBIQ_ENABLE_VIDEOS=true` after checking your profile. These are discovery results, not permission to reuse media. We display metadata rather than automatically downloading media or embedding third-party content.

Image filters include aspect ratio and minimum size. Video filters include duration, minimum resolution, and discovery-date freshness. An absent summary or moments array simply means that metadata is unavailable.

In [ ]:
if enabled('images'):
    response = requests.post(
        'https://api.microsoft.ai/v3/search/images',
        headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
        json={
            'query': 'Microsoft datacenter renewable energy', 'maxResults': 3,
            'aspectRatio': 'wide', 'imageSize': 'large', 'safeSearch': 'strict',
        },
        timeout=(10, 90),
    )
    response.raise_for_status()
    images = response.json()
    print(json.dumps(images, indent=2, ensure_ascii=False))

In [ ]:
if enabled('videos'):
    response = requests.post(
        'https://api.microsoft.ai/v3/search/videos',
        headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
        json={
            'query': 'Microsoft Foundry agents tutorial', 'maxResults': 3,
            'duration': 'medium', 'resolution': '720p',
            'freshness': 'year', 'safeSearch': 'strict',
        },
        timeout=(10, 90),
    )
    response.raise_for_status()
    videos = response.json()
    print(json.dumps(videos, indent=2, ensure_ascii=False))

## 8. Places: structured local information (optional beta)

**Places is beta and requires profile enablement.** Set `WEBIQ_ENABLE_PLACES=true` only after confirming access. A flag cannot grant access; a profile without Places permission returns HTTP 403. No additional Azure resource is required.

Use `POST /search/places` to find places with structured addresses, coordinates, opening hours, provider reviews and photos when available. Unlike Web Search's `webResults`, the response uses **`placeResults`**, and a place is identified by `name`, not `title`.

| Parameter | What to explore |
|---|---|
| `query` | A place-search query, up to 1,000 characters |
| `maxResults` | Between 1 and 10; fewer matches may be returned |
| `language`, `region` | Interface language and market context |
| `location` | Optional geographic context as `lat:<latitude>;long:<longitude>`; otherwise location is inferred from the query |

This exercise searches for public libraries near downtown Seattle. Compare the structured response with the passages returned by Web Search.

In [ ]:
if enabled('places'):
    response = requests.post(
        'https://api.microsoft.ai/v3/search/places',
        headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
        json={
            'query': 'public libraries near downtown Seattle',
            'maxResults': 3, 'language': 'en', 'region': 'US',
            'location': 'lat:47.6062;long:-122.3321',
        },
        timeout=(10, 90),
    )
    response.raise_for_status()
    places = response.json()
    print(json.dumps(places, indent=2, ensure_ascii=False))

### Inspect a place without inventing missing data

`location` contains `addressLine`, `city`, `region`, `country`, `postalCode`, `latitude` and `longitude` when available. `openHours.standard` maps weekdays to arrays of time ranges; `openHours.exceptions` can describe date-specific closures or hours. Also check `closureState` before interpreting a weekly schedule.

Ratings and photos can come from different providers. Preserve their attribution, do not treat missing ratings as zero, and do not assume a photo may be reused. Opening hours can change; verify important details with the place's own website. The next cell makes **no additional API request**.

In [ ]:
if enabled('places'):
    print('Places returned:', len(places.get('placeResults', [])))
    for place in places.get('placeResults', []):
        print('\nName:', place.get('name', '(unavailable)'))
        print('Website:', place.get('businessUrl', '(unavailable)'))
        print('Address and coordinates:', place.get('location', {}))
        print('Closure state:', place.get('closureState', '(unavailable)'))
        print('Opening hours:', place.get('openHours', {}))
        for review in place.get('reviews', []):
            print('Rating provider:', review.get('providerName', '(unavailable)'))
            print('Rating:', review.get('ratingScore', '(unavailable)'))
            print('Rating count:', review.get('ratingCount', '(unavailable)'))

**Checkpoint:** choose one library. Which structured fields are present? Are opening-hour exceptions provided? Which facts would you confirm on the library's own website before planning a visit?

**Exercise:** change the query and coordinates to another city, then rerun the Places request and inspection cells. Do not assume that `location` is a strict radius filter or that results are sorted by distance; neither is promised by these parameters. Note any missing fields rather than filling them in.

Reference: [Places REST API (Beta)](https://learn.microsoft.com/web-iq/reference/places).

## 9. Auto: typed evidence with scoped citations (optional beta)

**Auto is beta and requires explicit AppID enablement.** Set `WEBIQ_ENABLE_AUTO=true` only if enabled. Its response is **not** a Web Search result: `results` contains typed `evidence` and citations supporting only that evidence item. The endpoint may change; do not assume generated text is supported until you inspect its citations.

Use `POST /auto` when you want typed evidence across eligible answer categories rather than calling a single dedicated search API. It supplies evidence for your application or model, not a substitute for checking sources.

| Parameter | What to explore |
|---|---|
| `query` | A question or search query, 1-1,000 characters |
| `mode` | `fast` prioritizes speed; `advanced` prioritizes comprehensiveness |
| `responseFilter` | Eligible categories: `web`, `news`, `timezone`, `weather`, `images`, `videos`; eligibility does not guarantee every category appears |
| `maxResponseLength` | Total response-length budget, unlike Web Search's per-result `maxLength`; values are clamped to 1,500-500,000 |
| `language`, `region`, `location` | Language, market and optional geographic context |

Start with the same question as Web Search and restrict eligible evidence to Web and News. `places` is **not** a documented Auto response filter; use the dedicated Places endpoint for structured place information.

In [ ]:
if enabled('auto'):
    started = time.perf_counter()
    response = requests.post(
        'https://api.microsoft.ai/v3/auto',
        headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
        json={
            'query': question, 'mode': 'fast', 'maxResponseLength': 6000,
            'responseFilter': ['web', 'news'], 'language': 'en', 'region': 'US',
        },
        timeout=(10, 90),
    )
    response.raise_for_status()
    auto = response.json()
    auto_fast_seconds = time.perf_counter() - started
    print(f'Fast mode: {auto_fast_seconds:.2f}s')
    print(json.dumps(auto, indent=2, ensure_ascii=False))

### Read each evidence item with its own citations

The discriminator `evidence.type` tells you how to interpret `evidence.content`:

| Type | Content |
|---|---|
| `text` | Text evidence for Web or News |
| `media/image`, `media/video` | Structured media result |
| `structured/weather`, `structured/timeZone` | Structured weather or time-zone information |

Each result's `citations` belongs only to **that result**. Keep that pairing; do not flatten all citations into a pool and imply every source supports every item. Citation fields can include `contentSource`, `title`, `url`, `lastUpdatedAt` and `crawledAt`. Missing dates remain unknown.

The next cell inspects the existing response without another request. An empty or omitted `results` collection means no evidence was returned, not permission to invent an answer. An evidence item without citations should not be treated as independently verified.

In [ ]:
if enabled('auto'):
    print('Evidence items:', len(auto.get('results', [])))
    for index, item in enumerate(auto.get('results', []), 1):
        evidence = item.get('evidence', {})
        print(f"\n[{index}] Type: {evidence.get('type', '(unavailable)')}")
        print('Content:', json.dumps(evidence.get('content'), ensure_ascii=False))
        citations = item.get('citations', [])
        print('Supporting citations:', len(citations))
        for citation in citations:
            print('Source:', citation.get('contentSource', '(unavailable)'))
            print('Title:', citation.get('title', '(unavailable)'))
            print('URL:', citation.get('url', '(unavailable)'))
            print('Updated:', citation.get('lastUpdatedAt', '(unavailable)'))
            print('Crawled:', citation.get('crawledAt', '(unavailable)'))

### Optional: compare fast and advanced modes

Set `RUN_AUTO_ADVANCED=True` for one additional request using the **same query, filters and length budget** as the fast example. Compare elapsed time, evidence coverage and the usefulness of citations. One request per mode is a workshop observation, not a performance benchmark; caching, load and changing source coverage can affect results. Advanced mode does not guarantee more evidence or better accuracy on every query.

After comparing, optionally change `responseFilter` to `['weather', 'timezone']` and ask about Seattle's weather and local time, including `location='lat:47.6062;long:-122.3321'` in the JSON body. Inspect the actual `evidence.type` rather than assuming text. Only run categories available to your profile.

In [ ]:
RUN_AUTO_ADVANCED = False
if RUN_AUTO_ADVANCED and enabled('auto'):
    started = time.perf_counter()
    response = requests.post(
        'https://api.microsoft.ai/v3/auto',
        headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
        json={
            'query': question, 'mode': 'advanced', 'maxResponseLength': 6000,
            'responseFilter': ['web', 'news'], 'language': 'en', 'region': 'US',
        },
        timeout=(10, 90),
    )
    response.raise_for_status()
    auto_advanced = response.json()
    auto_advanced_seconds = time.perf_counter() - started
    print(f'Fast: {auto_fast_seconds:.2f}s; advanced: {auto_advanced_seconds:.2f}s')
    print('Fast evidence items:', len(auto.get('results', [])))
    print('Advanced evidence items:', len(auto_advanced.get('results', [])))
    print(json.dumps(auto_advanced, indent=2, ensure_ascii=False))

**Checkpoint:** choose one Auto evidence item and open its supporting URLs. Do they support the content? Are the sources primary or secondary? What uncertainty remains? Compare this evidence with Web Search's passages for the same question.

**Exercise:** explain when you would use a dedicated API for explicit retrieval control, when you would use Auto for typed evidence, and when you would add an agent to synthesize an answer. Keep the distinction between a retrieved passage, an evidence item and a final answer clear.

Reference: [Auto REST API (Beta)](https://learn.microsoft.com/web-iq/reference/auto).

## 10. Challenge: build an evidence-backed briefing

Choose a public topic relevant to your audience. Retrieve three sources and write a three-claim briefing with a URL supporting each claim. Identify a missing date or conflicting claim rather than filling it in. If Browse is available, check whether the full source changes your interpretation of its passage.

**Acceptance criteria:** three source-backed claims, no invented dates, one stated uncertainty, and an explanation of why you chose Web Search, News or Browse. If entitled to Places or Auto, include a structured place fact or an Auto evidence item, retaining its provider/source attribution. No LLM is required for this exercise; lab 02 adds agent orchestration.

Change `question` and rerun only the relevant request cells. Do not paste confidential customer or internal project details into public search queries.

## Continue to lab 02

Open `02-foundry-web-iq-agent.ipynb` to let a Foundry agent call this same service through MCP. Notice the distinction: REST gives you explicit retrieval control; the agent adds model-driven tool selection, approvals and synthesis.

Before sharing this notebook, clear all outputs. Returned web content, trace IDs and URLs may still reveal the topics you searched for. Follow the root README cleanup steps after the workshop.

References: [REST quickstart](https://learn.microsoft.com/web-iq/get-started/quickstart), [Web Search](https://learn.microsoft.com/web-iq/reference/web-search), [Browse](https://learn.microsoft.com/web-iq/reference/browse), [Places beta](https://learn.microsoft.com/web-iq/reference/places), [Auto beta](https://learn.microsoft.com/web-iq/reference/auto).